In [ ]:
# !pip install neo4j


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import sys
import os

# Go up one level from notebooks/ to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

In [5]:
import pandas as pd

from src.database.neo4j_queries import *


In [ ]:
# from neo4j import GraphDatabase

# URI = "neo4j://127.0.0.1:7687"
# USERNAME = "neo4j"
# PASSWORD = "password"

# driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

In [7]:
df = pd.read_csv("../dataset/03_cleaned/cleaned_restaurant.csv")

df.head()

,url,name,category,rating,reviews,price,address,status,snippet,img,postal,lon,lat,nearest_mrt,name_norm,address_norm,category_clean,category_group
0,https://www.google.com/maps/place/$1.50+Dim+Su...,$1.50 Dim Sum Bedok,Hawker Stall,3.7,(479),$1.50,"Block 123 Bedok North Street 2, 01-142",Open 24 hours,"""Very reasonable price n very delicious""",https://lh3.googleusercontent.com/gps-cs-s/AHV...,460123.0,103.937343,1.329199,"['bedok_north', 'bedok_reservoir']",$1.50 DIM SUM BEDOK,"BLOCK 123 BEDOK NORTH STREET 2, 01-142",hawker stall,hawker
1,https://www.google.com/maps/place/%2301-12+Han...,#01-12 Handmade Noodles,Hawker Stall,4.4,(12),$1–10,"4A Eunos Cres, #01-99 4A",Hawker Stall,"""It tastes not bad at all, extra crispy one si...",https://lh3.googleusercontent.com/gps-cs-s/AHV...,402004.0,103.904256,1.320331,['eunos'],#01-12 HANDMADE NOODLES,"4A EUNOS CRES, #01-99 4A",hawker stall,hawker
2,https://www.google.com/maps/place/%2301-18+%E5...,#01-18 廣東砂煲飯。小吃,Hawker Stall,4.0,(5),$1–10,"#01-18 Geylang Bahru, Market & Food Centre",Hawker Stall,"""Very authentic and good claypot rice.""",https://lh3.googleusercontent.com/gps-cs-s/AHV...,330069.0,103.870005,1.321463,['geylang_bahru'],#01-18 廣東砂煲飯。小吃,"#01-18 GEYLANG BAHRU, MARKET & FOOD CENTRE",hawker stall,hawker
3,https://www.google.com/maps/place/%2301-20+%E8...,#01-20 萝卜糕 经济小吃,Hawker Stall,4.7,(11),$1–10,"22A Havelock Rd, #01-20",Closed · Opens 8 am,"""Fried till crispy yet not greasy.""",https://lh3.googleusercontent.com/gps-cs-s/AHV...,161022.0,103.829623,1.287971,['havelock'],#01-20 萝卜糕 经济小吃,"22A HAVELOCK RD, #01-20",hawker stall,hawker
4,https://www.google.com/maps/place/%2301-35+%E5...,#01-35 健康齋素食,Hawker Stall,5.0,(1),NaN,"79 Telok Blangah Dr, #01-35, Telok Blangah Foo...",Closed · Opens 6 am,Takeaway,NaN,100079.0,103.807618,1.273356,['telok_blangah'],#01-35 健康齋素食,"79 TELOK BLANGAH DR, #01-35, TELOK BLANGAH FOO...",hawker stall,hawker


In [8]:
def create_constraints(tx):
    tx.run("CREATE CONSTRAINT restaurant_name IF NOT EXISTS FOR (r:Restaurant) REQUIRE r.name IS UNIQUE")
    tx.run("CREATE CONSTRAINT mrt_name IF NOT EXISTS FOR (m:MRT) REQUIRE m.name IS UNIQUE")
    tx.run("CREATE CONSTRAINT category_name IF NOT EXISTS FOR (c:FoodCategory) REQUIRE c.name IS UNIQUE")

with driver.session() as session:
    session.execute_write(create_constraints)

In [9]:
def insert_data(tx, row):
    tx.run("""
        MERGE (r:Restaurant {name: $name})
        SET r.rating = $rating,
            r.reviews = $reviews,
            r.address = $address,
            r.lat = $lat,
            r.lon = $lon,
            r.price = $price

        MERGE (m:MRT {name: $mrt})

        MERGE (c:FoodCategory {name: $category})

        MERGE (r)-[:IS_NEAR_TO]->(m)
        MERGE (r)-[:FOOD_CATEGORIZED_AS]->(c)
    """,
    name=row["name"],
    rating=row["rating"],
    reviews=row["reviews"],
    address=row["address"],
    lat=row["lat"],
    lon=row["lon"],
    price=row["price"],
    mrt=row["nearest_mrt"],
    category=row["category_clean"]
    )

In [7]:
# with driver.session() as session:
#     for _, row in df.iterrows():
#         session.execute_write(insert_data, row)

In [16]:
search_restaurants("['kent_ridge']", "japanese")

[{'name': 'UDON DON BAR',
  'rating': 4.1,
  'address': '1 Create Way #01-12 Town Plaza'},
 {'name': 'Waa Cow!', 'rating': 4.7, 'address': '2 College Ave W, #01-06'}]

In [17]:
get_graph_summary()

{'total_nodes': 9446,
 'restaurants': 8009,
 'mrt_stations': 1142,
 'categories': 295,
 'relationships': [{'type': 'IS_NEAR_TO', 'count': 8363},
  {'type': 'FOOD_CATEGORIZED_AS', 'count': 8120}]}

In [18]:
# preview_nodes("Restaurant", 5)
preview_nodes("MRT", 5)
# preview_nodes("FoodCategory", 5)

[<Node element_id='4:ac4faac0-f38c-4002-9808-3ed919f7742f:1' labels=frozenset({'MRT'}) properties={'name': "['bedok_north', 'bedok_reservoir']"}>,
 <Node element_id='4:ac4faac0-f38c-4002-9808-3ed919f7742f:4' labels=frozenset({'MRT'}) properties={'name': "['eunos']"}>,
 <Node element_id='4:ac4faac0-f38c-4002-9808-3ed919f7742f:6' labels=frozenset({'MRT'}) properties={'name': "['geylang_bahru']"}>,
 <Node element_id='4:ac4faac0-f38c-4002-9808-3ed919f7742f:8' labels=frozenset({'MRT'}) properties={'name': "['havelock']"}>,
 <Node element_id='4:ac4faac0-f38c-4002-9808-3ed919f7742f:10' labels=frozenset({'MRT'}) properties={'name': "['telok_blangah']"}>]

In [19]:
check_isolated_restaurants()

[]